In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df = pd.read_csv("D:/My Resume/IBM_TELECOM/telcom/telco_clean.csv")
os.makedirs("reports/figures", exist_ok=True)
sns.set_theme(style="whitegrid", context="talk")
PALETTE = {0: "#9aa5b1", 1: "#d1495b"}  # stayed = grey, churned = red

In [8]:
def churn_rate_by(col, order=None):
    """Churn rate (%) and customer count for each value of a column."""
    out = (df.groupby(col)["churn_value"]
             .agg(customers="count", churn_rate=lambda s: s.mean() * 100)
             .reset_index())
    if order:
        out[col] = pd.Categorical(out[col], categories=order, ordered=True)
        out = out.sort_values(col)
    return out

In [9]:
def bar_chart(col, title, fname, order=None):
    data = churn_rate_by(col, order)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=data, x=col, y="churn_rate", color="#d1495b", ax=ax)
    overall = df["churn_value"].mean() * 100
    ax.axhline(overall, ls="--", color="black", lw=1)
    ax.text(0.99, overall + 1, f"Overall {overall:.1f}%", ha="right",
            transform=ax.get_yaxis_transform(), fontsize=11)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=11)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Churn rate (%)")
    plt.xticks(rotation=15)
    plt.tight_layout()
    fig.savefig(f"reports/figures/{fname}.png", dpi=150)
    plt.close(fig)
    print(f"\n{title}\n", data.round(1).to_string(index=False))

In [10]:
# 1. Overall churn (class balance)
print("Overall churn rate: {:.1f}%".format(df["churn_value"].mean() * 100))

Overall churn rate: 26.5%


In [11]:
# 2. Churn rate by key categorical drivers

bar_chart("contract", "Month-to-month customers churn far more", "01_churn_by_contract",
          ["Month-to-month", "One year", "Two year"])
bar_chart("tenurebucket", "Most churn happens in the first year", "02_churn_by_tenure",
          ["0-12", "13-24", "25-48", "49-72"])
bar_chart("internet_service", "Fiber optic customers churn the most", "03_churn_by_internet")
bar_chart("payment_method", "Electronic check users churn the most", "04_churn_by_payment")
bar_chart("tech_support", "Customers without tech support churn more", "05_churn_by_tech_support")


Month-to-month customers churn far more
       contract  customers  churn_rate
Month-to-month       3875        42.7
      One year       1473        11.3
      Two year       1695         2.8

Most churn happens in the first year
 tenurebucket  customers  churn_rate
        0-12       2186        47.4
       13-24       1024        28.7
       25-48       1594        20.4
       49-72       2239         9.5

Fiber optic customers churn the most
 internet_service  customers  churn_rate
             DSL       2421        19.0
     Fiber optic       3096        41.9
              No       1526         7.4

Electronic check users churn the most
            payment_method  customers  churn_rate
Bank transfer (automatic)       1544        16.7
  Credit card (automatic)       1522        15.2
         Electronic check       2365        45.3
             Mailed check       1612        19.1

Customers without tech support churn more
 tech_support  customers  churn_rate
          No       4999

In [12]:
# 3. Numeric drivers: monthly charges and tenure

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(axes, ["monthly_charges", "tenure_months"],
                          ["Monthly charges ($)", "Tenure (months)"]):
    sns.kdeplot(data=df, x=col, hue="churn_value", palette=PALETTE,
                fill=True, common_norm=False, alpha=0.4, ax=ax)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel("")
    ax.legend(labels=["Churned", "Stayed"], title="")
plt.tight_layout()
fig.savefig("reports/figures/06_numeric_distributions.png", dpi=150)
plt.close(fig)
 
print("\nAverages by churn status:")
print(df.groupby("churn_value")[["tenure_months", "monthly_charges", "total_charges", "cltv"]]
        .mean().round(1))


Averages by churn status:
             tenure_months  monthly_charges  total_charges    cltv
churn_value                                                       
0                     37.6             61.3         2549.9  4490.9
1                     18.0             74.4         1531.8  4149.4


In [13]:
# 4. Top churn reasons

reasons = (df[df["churn_value"] == 1]["churn_reason"].value_counts().head(10)
           .sort_values())
fig, ax = plt.subplots(figsize=(11, 6))
reasons.plot(kind="barh", color="#d1495b", ax=ax)
ax.set_title("Top 10 reasons customers left", loc="left", fontweight="bold")
ax.set_xlabel("Churned customers")
plt.tight_layout()
fig.savefig("reports/figures/07_top_churn_reasons.png", dpi=150)
plt.close(fig)

In [14]:
df.dtypes

customer_id           object
city                  object
zip_code               int64
lat_long              object
latitude             float64
longitude            float64
gender                object
senior_citizen        object
partner               object
dependents            object
tenure_months          int64
phone_service         object
multiple_lines        object
internet_service      object
online_security       object
online_backup         object
device_protection     object
tech_support          object
streaming_tv          object
streaming_movies      object
contract              object
paperless_billing     object
payment_method        object
monthly_charges      float64
total_charges        float64
churn_label           object
churn_value            int64
churn_score            int64
cltv                   int64
churn_reason          object
tenurebucket          object
numservices            int64
dtype: object

In [15]:
# 5. Correlation with churn (numeric features)

num = df[["churn_value", "tenure_months", "monthly_charges", "total_charges",
          "cltv", "numservices"]]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation of numeric features", loc="left", fontweight="bold")
plt.tight_layout()
fig.savefig("reports/figures/08_correlation.png", dpi=150)
plt.close(fig)

In [16]:
# 6. Headline numbers for the README and resume

churned = df[df["churn_value"] == 1]
seg = df[(df["contract"] == "Month-to-month") & (df["tenure_months"] <= 12)]
print("\n--- Headline numbers ---")
print(f"Customers: {len(df):,} | churned: {len(churned):,} ({df['churn_value'].mean():.1%})")
print(f"Monthly revenue lost to churn: ${churned['monthly_charges'].sum():,.0f} "
      f"({churned['monthly_charges'].sum() / df['monthly_charges'].sum():.1%} of total)")
print(f"Annualized revenue lost: ${churned['monthly_charges'].sum() * 12:,.0f}")
print(f"CLTV of churned customers: ${churned['cltv'].sum():,.0f}")
print(f"New month-to-month segment: {len(seg):,} customers, "
      f"{seg['churn_value'].mean():.1%} churn, "
      f"{seg['churn_value'].sum() / len(churned):.1%} of all churn")
print("Charts saved in reports/figures/")


--- Headline numbers ---
Customers: 7,043 | churned: 1,869 (26.5%)
Monthly revenue lost to churn: $139,131 (30.5% of total)
Annualized revenue lost: $1,669,570
CLTV of churned customers: $7,755,256
New month-to-month segment: 1,994 customers, 51.4% churn, 54.8% of all churn
Charts saved in reports/figures/
